In [1]:
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType
import re
from functools import reduce
from operator import add

In [2]:
from pyspark.sql import SparkSession
from src.utils.logger import get_logger
import src.utils.config as config 


print(f"DEBUG: Access Key is {config.MINIO_ACCESS_KEY[:3]} + ***") 
print(f"DEBUG: ENDPOINT is {config.MINIO_ENDPOINT}")

# Get the container's hostname dynamically

logger = get_logger(__name__)

def create_spark_session(app_name: str) -> SparkSession:
    """
    Creates and returns a configured Spark session for MinIO.
    """
    logger.info(f"Creating Spark Session: {app_name}")
    
    # This pulls the necessary S3A connectors from Maven Central
    
    

    spark = (   
        SparkSession.builder
        .appName(app_name)
        .master("spark://spark-master:7077")
        .config("spark.driver.host", config.DRIVER_HOST)
        .config("spark.driver.bindAddress", "0.0.0.0")
        .config("spark.driver.port", config.SPARK_DRIVER_PORT)
        .config("spark.driver.blockManager.port", config.SPARK_BLOCK_MANAGER_PORT)
        .config("spark.sql.shuffle.partitions", "50")
        .config("spark.executor.instances", "1") # adjust based on resources
        .config("spark.executor.cores", "2")
        .config("spark.sql.adaptive.coalescePartitions.enabled", "false")
        #.config("spark.executor.memory", "2g") # adjust based on resources
        #.config("spark.driver.memory", "2g") # adjust based on resources

         # hadoop S3A Configuration
      
        .config("spark.hadoop.fs.s3a.endpoint", config.MINIO_ENDPOINT)
        .config("spark.hadoop.fs.s3a.access.key", config.MINIO_ACCESS_KEY)
        .config("spark.hadoop.fs.s3a.secret.key", config.MINIO_SECRET_KEY)
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", str(config.MINIO_SECURE).lower())
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") # prevent class resolution
        .config("spark.sql.caseSensitive", "false")
        .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
        .config("spark.cores.max", "2")
        .config("spark.driver.extraJavaOptions", "-Djava.net.preferIPv4Stack=true")
        .config("spark.executor.extraJavaOptions", "-Djava.net.preferIPv4Stack=true")
        .config("spark.hadoop.fs.s3a.fast.upload", "true") #performance improvement
        .config("spark.sql.files.maxPartitionBytes", "16777216") #16MB partitions
        #.config("spark.network.timeout", "1200s")
        #.config("spark.rpc.askTimeout", "600s")
        #.config("spark.executor.heartbeatInterval", "120s")
        #.config("spark.hadoop.fs.s3a.connection.timeout", "600000")
        #.config("spark.hadoop.fs.s3a.paging.maximum", "1000")
        
        .getOrCreate()
    )
    # Suppress verbose logs
    spark.sparkContext.setLogLevel("WARN")
    logger.info("Spark session created successfully")
    return spark

[CONFIG] Stage: transform
[CONFIG] Loaded env: /opt/spark-app/env/.env.transform
[CONFIG] MinIO Endpoint: lakehouse-minio:9000
[CONFIG] Access Key (masked): tra***
DEBUG: Access Key is tra + ***
DEBUG: ENDPOINT is lakehouse-minio:9000


In [3]:
spark= create_spark_session("notebook_test")
Spain_24= spark.read.parquet("s3a://bronze/SPAIN/2024_BRONZE/")

2026-07-23 17:17:55 | INFO | lakehouse.__main__ | Creating Spark Session: notebook_test


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/23 17:18:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


2026-07-23 17:21:12 | INFO | lakehouse.__main__ | Spark session created successfully


26/07/23 17:22:18 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [4]:
Spain_24.rdd.getNumPartitions()

5

In [5]:
Spain_24

DataFrame[beneficiario: string, grupo_empresa: string, provincia: string, municipio: string, medida: string, objetivo_esp: string, fec_ini: string, fec_fin: string, feaga: string, feader: string, importecofin: string, feader_cofin: string, importe_euros: string, source_country: string, source_year: string, ingested_at: timestamp]

In [6]:
Spain_24.printSchema()

root
 |-- beneficiario: string (nullable = true)
 |-- grupo_empresa: string (nullable = true)
 |-- provincia: string (nullable = true)
 |-- municipio: string (nullable = true)
 |-- medida: string (nullable = true)
 |-- objetivo_esp: string (nullable = true)
 |-- fec_ini: string (nullable = true)
 |-- fec_fin: string (nullable = true)
 |-- feaga: string (nullable = true)
 |-- feader: string (nullable = true)
 |-- importecofin: string (nullable = true)
 |-- feader_cofin: string (nullable = true)
 |-- importe_euros: string (nullable = true)
 |-- source_country: string (nullable = true)
 |-- source_year: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



In [7]:
# let's know the number of non null or nan values present in each column

# 1. Build expressions dynamically based on each column's specific data type
count_expressions = []


for col_name, col_type in Spain_24.dtypes:
    # Base condition: works for ALL data types (Timestamps, Strings, Ints, etc.)
    condition = F.col(col_name).isNotNull()

    # Only append the NaN check if the column is a floating-point numeric type
    if col_type in ("double", "float"):
        condition = condition & ~F.isnan(F.col(col_name))
    
    # Rule B: Catch empty text cells in string columns
    elif col_type == "string":
        condition = condition & ~(F.trim(F.col(col_name))).isin("", "N/A", "n/a", "NA", "na")
    
    # Aggregate using the safe conditional block
    count_expressions.append(F.count(F.when(condition, 1)).alias(col_name))

# 2. Run the single, optimized aggregation across the cluster
counts_row = Spain_24.select(count_expressions).first()

print(counts_row)


Row(beneficiario=2217849, grupo_empresa=4082, provincia=2217849, municipio=2217849, medida=2217849, objetivo_esp=2089432, fec_ini=6383, fec_fin=4074, feaga=2217848, feader=2217849, importecofin=2217849, feader_cofin=2217849, importe_euros=2217849, source_country=2217849, source_year=2217849, ingested_at=2217849)


In [8]:
# Count total rows in the DataFrame
total_rows = Spain_24.count()

# Print header
print(f'{"Column Name": <65} | {"Missing Percentage"}')
print("-" * 85)

# Calculate and print missing percentage for each column
for column, valid_count in counts_row.asDict().items():
    missing_percentage = ((total_rows - valid_count) / total_rows) * 100
    print(f"{column: <65} | {missing_percentage: >10.3f}%")


Column Name                                                       | Missing Percentage
-------------------------------------------------------------------------------------
beneficiario                                                      |      0.000%
grupo_empresa                                                     |     99.816%
provincia                                                         |      0.000%
municipio                                                         |      0.000%
medida                                                            |      0.000%
objetivo_esp                                                      |      5.790%
fec_ini                                                           |     99.712%
fec_fin                                                           |     99.816%
feaga                                                             |      0.000%
feader                                                            |      0.000%
importecofin               

In [11]:
Spain_24_cleaned= Spain_24.select(
    F.col("beneficiario").alias("beneficiary"),
    F.col("municipio").alias("municipality"),
    F.col("provincia").alias("province"),
    F.col("source_country").alias("country"),
    F.col("source_year").alias("year"),
    F.col("medida").alias("intervention_code"),
    F.regexp_replace(F.col("feaga"), ",", ".").cast(DoubleType()).alias("total_eagf_income_support"),
    F.regexp_replace(F.col("feader"), ",", ".").cast(DoubleType()).alias("total_eafrd_income_support"),
    F.regexp_replace(F.col("importecofin"), ",", ".").cast(DoubleType()).alias("national_cofunding_amount")
    
).fillna(0.0, subset=["total_eagf_income_support", "total_eafrd_income_support", "national_cofunding_amount"])

# fill nulls in text fields
Spain24_cleaned= Spain_24_cleaned.fillna("UNKNOWN", subset= ["beneficiary", "municipality", "intervention_code"])

In [12]:
Spain24_cleaned.show(5, truncate=False)

+-------------------------------------------------+-------------+--------+-------+----+-----------------------------------------------------------------------------------+-------------------------+--------------------------+-------------------------+
|beneficiary                                      |municipality |province|country|year|intervention_code                                                                  |total_eagf_income_support|total_eafrd_income_support|national_cofunding_amount|
+-------------------------------------------------+-------------+--------+-------+----+-----------------------------------------------------------------------------------+-------------------------+--------------------------+-------------------------+
|: HERENCIA YACENTE DE DON VICTORIO ARTALEJO LOPEZ|28982 - Parla|Madrid  |SPAIN  |2024|II.1   Régimen de pago básico                                                      |24217.98                 |0.0                       |0.0                    

In [13]:
spark.stop()

### WE would be implementing star schema for our database schema

In [ ]:
# Phase 2 : Dim_Beneficiary
dim_beneficiary= 

In [ ]:
Spain24_cleaned.select(
    "eagf_amount",
    "eafrd_amount",
    "spainsh_cofunding_amount",
    "eafrd_and_cofunding_amount",
    "total_amount"
    
).limit(12).show()

In [ ]:
Spain24_salted= Spain24_cleaned.withColumn("salt", F.floor(F.rand()*10))
# Step 1: Perform the GroupBy and correctly aggregate using F.sum and F.col
spain24_stage1 = Spain24_salted.groupBy(
   "salt",  "beneficiary", "intervention_code"
).agg(
    # EAGF merge (Summing to capture all records per beneficiary)
    F.sum(F.col("eagf_amount")).alias("total_eagf_income_support"),
    
    # EAFRD merge
    F.sum(F.col("eafrd_amount")).alias("total_eafrd_income_support"),

    # Co-Financing Merge
    F.sum(F.col("spainsh_cofunding_amount")).alias("total_spanish_co_funding"),
    
    #add metadata
    F.first("source_country", ignorenulls=True).alias("country"),
    
    F.first("source_year", ignorenulls=True).alias("year"),

    F.first("municipality", ignorenulls=True).alias("municipality")

    
)


In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", "50")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "false")

In [ ]:
spain24_stage1_repartitioned= spain24_stage1.repartition(50, "beneficiary")
spain24_merged= spain24_stage1_repartitioned.groupBy(
     "beneficiary", "intervention_code"
).agg(
    # EAGF merge (Summing to capture all records per beneficiary)
    F.sum(F.col("total_eagf_income_support")).alias("total_eagf_income_support"),
    
    # EAFRD merge
    F.sum(F.col("total_eafrd_income_support")).alias("total_eafrd_income_support"),

    # Co-Financing Merge
    F.sum(F.col("total_spanish_co_funding")).alias("total_spanish_co_funding"),

    #add metadata
    F.first("country", ignorenulls=True).alias("country"),
    
    F.first("year", ignorenulls=True).alias("year"),

    F.first("municipality", ignorenulls=True).alias("municipality")
)



In [ ]:
spain24_merged.explain()

In [ ]:
spain24_merged.select(
    "beneficiary", "intervention_code",
    "total_eagf_income_support"
).limit(3).show()

In [ ]:
# Step 2: Add your calculated grand total column directly to the merged DataFrame
# Note: True EU Payout = EAGF + EAFRD
spain24_merged = spain24_merged \
    .withColumn(
       "grand_total_eu_payout",
       F.coalesce(F.col("total_eagf_income_support"), F.lit(0.0)) + F.coalesce(F.col("total_eafrd_income_support"), F.lit(0.0))
    ) \
    .withColumn(
        "grand_total_public_payout",
        F.coalesce(F.col("total_eagf_income_support"), F.lit(0.0)) + F.coalesce(F.col("total_eafrd_income_support"), F.lit(0.0)) +F.coalesce(F.col("total_spanish_co_funding"), F.lit(0.0)) 
        
    )

# Step 3: Now define your complete numeric list, including the new calculated column
final_numeric_columns = [
    "total_eagf_income_support",
    "total_eafrd_income_support",
    "total_spanish_co_funding",
    "grand_total_eu_payout",
    "grand_total_public_payout",
]

# Step 4: Finalize the revamped DataFrame by filling nulls on the fully computed dataset
spain24_revamped = spain24_merged.na.fill(0.0, subset=final_numeric_columns)




In [ ]:
# Verify the results
spain24_revamped.limit(5).show(truncate=False)

In [ ]:
# Filter out null beneficiaries
beneficiary_key= 'beneficiary'
scheme_key= 'intervention_code'
municipality='municipality'

# Filter out null beneficiaries
spain24_core= spain24_revamped.filter(F.col(scheme_key).isNotNull())

# Step 1: Clean and fetch distinct schemes using PySpark natively
distinct_schemes_df= spain24_core \
     .select(scheme_key) \
     .filter(~F.col(scheme_key).isin(r"N\a", r"n\a", "")) \
     .dropna() \
     .distinct()

# Step 2: Bring only the final, clean list over to the driver
distinct_schemes = [row[0] for row in distinct_schemes_df.collect()]

print(f" Distinct schemes in intervention_sector code are :{distinct_schemes}")


In [ ]:
anchors= [beneficiary_key]

# Drop dead database rows using grand_total_payout
spain24_filtered= spain24_revamped.filter(
    ~(F.col(grand_total_public_payout)=0.0)
)

## Macro pillar aggregration

In [ ]:
Scheme_Registry = {
    # 1. Direct Income Support
    "I.1   Ayuda básica a la renta para la sostenibilidad": {
        "english_translation": "Basic Income Support for Sustainability",
        "group_name": "Direct Income Support",
        "canonical_slug": "direct_payment_biss",
    },
    "I.2   Ayuda redistributiva complementaria a la renta para la sostenibilidad": {
        "english_translation": "Complementary Redistributive Income Support for Sustainability",
        "group_name": "Direct Income Support",
        "canonical_slug": "redistributive_support",
    },
    "I.3   Ayuda complementaria a la renta para jóvenes agricultores": {
        "english_translation": "Complementary Income Support for Young Farmers",
        "group_name": "Direct Income Support",
        "canonical_slug": "young_farmer_income",
    },
    "I.6   Ayuda a la renta asociada": {
        "english_translation": "Coupled Income Support",
        "group_name": "Direct Income Support",
        "canonical_slug": "coupled_income_support",
    },
    "I.7   Pago específico al cultivo del algodón": {
        "english_translation": "Crop-Specific Payment for Cotton",
        "group_name": "Direct Income Support",
        "canonical_slug": "cotton_payment",
    },
    "II.1   Régimen de pago básico": {
        "english_translation": "Basic Payment Scheme",
        "group_name": "Direct Income Support",
        "canonical_slug": "direct_payment_biss",
    },
    "II.6   Pago para los jóvenes agricultores": {
        "english_translation": "Payment for Young Farmers",
        "group_name": "Direct Income Support",
        "canonical_slug": "young_farmer_income",
    },
    "II.7   Ayuda asociada voluntaria": {
        "english_translation": "Voluntary Coupled Support",
        "group_name": "Direct Income Support",
        "canonical_slug": "coupled_income_support",
    },
    "II.8   Pago específico al cultivo de algodón": {
        "english_translation": "Crop-Specific Payment for Cotton",
        "group_name": "Direct Income Support",
        "canonical_slug": "cotton_payment",
    },
    "II.9   Régimen para los pequeños agricultores": {
        "english_translation": "Small Farmers Scheme",
        "group_name": "Direct Income Support",
        "canonical_slug": "small_farmers_scheme",
    },
    "II.10   Medidas establecidas en el anexo I del Reglamento 73/2009": {
        "english_translation": "Measures under Annex I of Regulation (EC) No 73/2009",
        "group_name": "Direct Income Support",
        "canonical_slug": "direct_support_legacy",
    },
    "II.11   Recuperaciones": {
        "english_translation": "Recoveries and Adjustments",
        "group_name": "Direct Income Support",
        "canonical_slug": "direct_support_legacy",
    },
    # 2. Sectoral & Market Support
    "III.1   En el sector de las frutas y hortalizas": {
        "english_translation": "Fruit and Vegetables Sector Interventions",
        "group_name": "Sectoral & Market Support",
        "canonical_slug": "fruit_veg_support",
    },
    "III.2   En el sector de los productos apícolas": {
        "english_translation": "Apiculture Sector Products",
        "group_name": "Sectoral & Market Support",
        "canonical_slug": "apiculture_support",
    },
    "III.3   En el sector vitivinícola": {
        "english_translation": "Wine Sector Interventions",
        "group_name": "Sectoral & Market Support",
        "canonical_slug": "wine_sector_support",
    },
    "IV.3   El programa escolar de la UE, los programas de fruta y de leche en los centros escolares": {
        "english_translation": "EU School Scheme for Fruit, Vegetables, and Milk",
        "group_name": "Sectoral & Market Support",
        "canonical_slug": "school_nutrition_program",
    },
    "IV.4   Reserva agrícola de crisis": {
        "english_translation": "Agricultural Crisis Reserve",
        "group_name": "Sectoral & Market Support",
        "canonical_slug": "crisis_reserve",
    },
    "IV.5   Ayuda en el sector de las frutas y hortalizas": {
        "english_translation": "Aid in the Fruit and Vegetables Sector",
        "group_name": "Sectoral & Market Support",
        "canonical_slug": "fruit_veg_support",
    },
    "IV.6   Ayuda en el sector vitivinícola": {
        "english_translation": "Aid in the Wine Sector",
        "group_name": "Sectoral & Market Support",
        "canonical_slug": "wine_sector_support",
    },
    "IV.8   Ayuda en el sector apícola": {
        "english_translation": "Aid in the Apiculture Sector",
        "group_name": "Sectoral & Market Support",
        "canonical_slug": "apiculture_support",
    },
    "VII.1   Medidas establecidas en el Reglamento 228/2013": {
        "english_translation": "Measures under Regulation (EU) No 228/2013 (POSEI Canary Islands)",
        "group_name": "Sectoral & Market Support",
        "canonical_slug": "outermost_regions_posei",
    },
    "IX.1   Medidas de información y de promoción": {
        "english_translation": "Information and Promotion Measures",
        "group_name": "Sectoral & Market Support",
        "canonical_slug": "market_food_promotion",
    },
    # 3. Agri-Environment, Climate & Animals
    "I.4   Regímenes en favor del clima y el medio ambiente": {
        "english_translation": "Schemes for Climate and Environment (Eco-schemes)",
        "group_name": "Agri-Environment & Climate",
        "canonical_slug": "eco_schemes",
    },
    "II.4   Pago para prácticas agrícolas beneficiosas para el clima y el medio ambiente": {
        "english_translation": "Payment for Climate-Beneficial Practices (Greening)",
        "group_name": "Agri-Environment & Climate",
        "canonical_slug": "eco_schemes",
    },
    "V.1   Compromisos medioambientales y climáticos y otros compromisos de gestión": {
        "english_translation": "Environmental, Climate, and Other Management Commitments",
        "group_name": "Agri-Environment & Climate",
        "canonical_slug": "agri_environmental_commitments",
    },
    "V.2   Zonas con limitaciones naturales u otras limitaciones específicas": {
        "english_translation": "Areas Facing Natural or Other Specific Constraints",
        "group_name": "Agri-Environment & Climate",
        "canonical_slug": "natural_constraints_anc",
    },
    "V.3   Desventajas específicas resultantes de determinados requisitos obligatorios": {
        "english_translation": "Specific Disadvantages from Mandatory Requirements",
        "group_name": "Agri-Environment & Climate",
        "canonical_slug": "natura_2000_disadvantages",
    },
    "VI.15   Medida agroambiental y climática": {
        "english_translation": "Agri-Environment-Climate Measure",
        "group_name": "Agri-Environment & Climate",
        "canonical_slug": "agri_environmental_commitments",
    },
    "VI.16   Agricultura ecológica": {
        "english_translation": "Organic Farming",
        "group_name": "Agri-Environment & Climate",
        "canonical_slug": "organic_farming",
    },
    "VI.17   Pagos al amparo de Natura 2000 y de la Directiva Marco del Agua": {
        "english_translation": "Payments under Natura 2000 and Water Framework Directive",
        "group_name": "Agri-Environment & Climate",
        "canonical_slug": "natura_2000_disadvantages",
    },
    "VI.18   Ayuda a zonas con limitaciones naturales u otras limitaciones específicas": {
        "english_translation": "Aid to Areas Facing Natural or Other Specific Constraints",
        "group_name": "Agri-Environment & Climate",
        "canonical_slug": "natural_constraints_anc",
    },
    "VI.19   Bienestar de los animales": {
        "english_translation": "Animal Welfare",
        "group_name": "Agri-Environment & Climate",
        "canonical_slug": "livestock_welfare_and_support",
    },
    # 4. Forestry & Silviculture
    "VI.8   Inversiones en el desarrollo de zonas forestales y la mejora de la viabilidad de los bosques": {
        "english_translation": "Investments in Forest Area Development and Viability",
        "group_name": "Forestry & Silviculture",
        "canonical_slug": "forestry_and_agroforestry",
    },
    "VI.9   Forestación y creación de superficies forestales": {
        "english_translation": "Afforestation and Creation of Woodland",
        "group_name": "Forestry & Silviculture",
        "canonical_slug": "forestry_and_agroforestry",
    },
    "VI.10   Implantación, regeneración o renovación de sistemas agroforestales": {
        "english_translation": "Establishment, Regeneration, or Renewal of Agroforestry Systems",
        "group_name": "Forestry & Silviculture",
        "canonical_slug": "forestry_and_agroforestry",
    },
    "VI.11   Prevención y reparación de los daños causados a los bosques por incendios, desastres naturales y catástrofes": {
        "english_translation": "Prevention and Repair of Forest Damage from Fires and Natural Disasters",
        "group_name": "Forestry & Silviculture",
        "canonical_slug": "forestry_and_agroforestry",
    },
    "VI.12   Inversiones para incrementar la capacidad de adaptación y el valor medioambiental de los ecosistemas forestales": {
        "english_translation": "Investments to Increase Forest Resilience and Environmental Value",
        "group_name": "Forestry & Silviculture",
        "canonical_slug": "forestry_and_agroforestry",
    },
    "VI.13   Inversiones en tecnologías forestales y en la transformación, movilización y comercialización de productos forestales": {
        "english_translation": "Investments in Forestry Technologies, Processing, and Marketing",
        "group_name": "Forestry & Silviculture",
        "canonical_slug": "forestry_and_agroforestry",
    },
    "VI.20   Servicios silvoambientales y climáticos y la conservación de los bosques": {
        "english_translation": "Silvo-Environmental and Climate Services and Forest Conservation",
        "group_name": "Forestry & Silviculture",
        "canonical_slug": "forestry_and_agroforestry",
    },
    # 5. Rural Development & Investments
    "22400 - Monzón": {
        "english_translation": "Monzón Regional Irrigation Infrastructure Project",
        "group_name": "Rural Development & Investments",
        "canonical_slug": "physical_capital_investments",
    },
    "M.113   Jubilación anticipada": {
        "english_translation": "Early Retirement Scheme",
        "group_name": "Rural Development & Investments",
        "canonical_slug": "early_retirement_legacy",
    },
    "V.4   Inversiones, incluidas las inversiones en infraestructuras de riego": {
        "english_translation": "Investments, Including Irrigation Infrastructure",
        "group_name": "Rural Development & Investments",
        "canonical_slug": "physical_capital_investments",
    },
    "V.5   Establecimiento de jóvenes agricultores, nuevos agricultores y puesta en marcha de nuevas empresas rurales": {
        "english_translation": "Setting Up of Young Farmers, New Farmers, and Rural Startups",
        "group_name": "Rural Development & Investments",
        "canonical_slug": "rural_business_startup",
    },
    "VI.3   Regímenes de calidad de los productos agrícolas y alimenticios": {
        "english_translation": "Quality Schemes for Agricultural Products and Foodstuffs",
        "group_name": "Rural Development & Investments",
        "canonical_slug": "quality_schemes",
    },
    "VI.4   Inversiones en activos físicos": {
        "english_translation": "Investments in Physical Assets",
        "group_name": "Rural Development & Investments",
        "canonical_slug": "physical_capital_investments",
    },
    "VI.5   Recuperación del potencial de producción agrícola dañado por desastres naturales e implantación de medidas preventivas adecuadas": {
        "english_translation": "Restoration of Agricultural Production Potential Damaged by Disasters",
        "group_name": "Rural Development & Investments",
        "canonical_slug": "agricultural_disaster_recovery",
    },
    "VI.6   Desarrollo de explotaciones agrícolas y empresas": {
        "english_translation": "Farm and Business Development",
        "group_name": "Rural Development & Investments",
        "canonical_slug": "rural_business_startup",
    },
    "VI.7   Servicios básicos y renovación de poblaciones en las zonas rurales": {
        "english_translation": "Basic Services and Village Renewal in Rural Areas",
        "group_name": "Rural Development & Investments",
        "canonical_slug": "village_renewal_infra",
    },
    "VI.14   Creación de agrupaciones y organizaciones de productores": {
        "english_translation": "Setting Up Producer Groups and Organizations",
        "group_name": "Rural Development & Investments",
        "canonical_slug": "producer_org_aid",
    },
    "VI.24   Ayuda para el desarrollo local de Leader (desarrollo local participativo)": {
        "english_translation": "Aid for LEADER Local Development (CLLD)",
        "group_name": "Rural Development & Investments",
        "canonical_slug": "leader_community_dev",
    },
    # 6. Knowledge, Innovation & Crisis
    "V.7   Cooperación": {
        "english_translation": "Cooperation Projects",
        "group_name": "Knowledge & Innovation",
        "canonical_slug": "cooperation_innovation",
    },
    "V.8   Intercambio de conocimientos y difusión de información": {
        "english_translation": "Knowledge Exchange and Information Dissemination",
        "group_name": "Knowledge & Innovation",
        "canonical_slug": "knowledge_transfer_training",
    },
    "V.9   Asistencia Técnica (calculado con la tasa media del Plan Estratégico PAC)": {
        "english_translation": "Technical Assistance (Strategic Plan Rate)",
        "group_name": "Technical Assistance",
        "canonical_slug": "technical_assistance_eafrd",
    },
    "VI.1   Transferencia de conocimientos y actividades de información": {
        "english_translation": "Knowledge Transfer and Information Actions",
        "group_name": "Knowledge & Innovation",
        "canonical_slug": "knowledge_transfer_training",
    },
    "VI.2   Servicios de asesoramiento, gestión y sustitución de explotaciones agrarias": {
        "english_translation": "Farm Advisory, Management, and Relief Services",
        "group_name": "Knowledge & Innovation",
        "canonical_slug": "farm_advisory_services",
    },
    "VI.21   Cooperación": {
        "english_translation": "Cooperation",
        "group_name": "Knowledge & Innovation",
        "canonical_slug": "cooperation_innovation",
    },
    "VI.22b   Ayuda temporal excepcional destinada a los agricultores y a las pymes especialmente afectados por la invasión de Ucrania por parte de Rusia": {
        "english_translation": "Exceptional Temporary Support (Ukraine War Impact)",
        "group_name": "Emergency Crisis Aid",
        "canonical_slug": "emergency_crisis_aid",
    },
    "VI.25   Asistencia técnica": {
        "english_translation": "Technical Assistance",
        "group_name": "Technical Assistance",
        "canonical_slug": "technical_assistance_eafrd",
    },
}

In [ ]:
anchors = ["beneficiary"]  
intervention_key = "intervention_sector_code"  # Your column containing raw Spanish strings

# --- 2. MACRO AGGREGATIONS ---
# Adjust column names based on your Spain dataset (e.g., FEAGA/FEADER or total payout)
macro_expressions = [
    F.sum("feaga_amount").alias("total_eagf_income_support"),
    F.sum("feader_amount").alias("total_eafrd_income_support"),
    F.sum("grand_total_public_payout").alias("grand_total_public_payout"),
]

spain24_macros = spain24_identifiable.groupBy(anchors).agg(*macro_expressions)


# --- 3. BUILD MAPPING EXPRESSION FROM NEW DICTIONARY STRUCTURE ---
mapping_expr = F.when(
    F.col(intervention_key).isNull(), "other_rural_development"
)

# Unpack key and nested dict from new Scheme_Registry
for raw_val, info in Scheme_Registry.items():
    slug = info["canonical_slug"]  # Extract slug from dictionary
    mapping_expr = mapping_expr.when(F.col(intervention_key) == raw_val, slug)

mapping_expr = mapping_expr.otherwise("other_rural_development")

# Add mapped scheme_token column
spain24_prepared = spain24_identifiable.withColumn(
    "scheme_token", mapping_expr
)


# --- 4. EXPLICIT PIVOT (MEMORY OPTIMIZED) ---
# Extract distinct slugs in Python to prevent Spark from running a 2nd pass
distinct_slugs = list(
    set([info["canonical_slug"] for info in Scheme_Registry.values()])
) + ["other_rural_development"]

spain24_pivoted_micros = (
    spain24_prepared.groupBy(anchors)
    .pivot("scheme_token", distinct_slugs)  # Explicit list prevents extra OOM-inducing scan
    .agg(F.sum("grand_total_public_payout"))
    .na.fill(0.0)
)


# --- 5. MASTER MERGE ---
spain24_master = spain24_macros.join(
    spain24_pivoted_micros, on=anchors, how="inner"
)

spain24_master.printSchema()

In [ ]:
spark.stop()